### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multipleschema types and methods for enforcing structured outputs. 

### Pydantic

Pydantic model provides the richest fetures set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002122D88D400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002122D88E120>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year in which movie has released")
    director:str=Field(description="Director name of that movie")
    rating:float=Field(description="The movie rating out of 5")

In [3]:
model_with_structured=model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002122D88D400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002122D88E120>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [4]:
model.invoke("Provide details about the movie Bahubali")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Bahubali. Let me start by recalling what I know. Bahubali is a 2015 Indian Telugu-language epic action film directed by S.S. Rajamouli. It\'s part of a duology, with Bahubali 2: The Conclusion released in 2017. The film stars Prabhas, Rana Daggubati, Anushka Shetty, Tamannaah Bhatia, and Sushant Singh Rajput. \n\nThe story is set in the kingdom of Mahishmati and revolves around the conflict between two rival factions within the kingdom. The main characters are Bahubali, the protagonist, and his brother Bhallaladeva. There\'s a significant backstory involving their father, Amarendra Baahubali, and their grandfather, Maharaja. The plot involves themes of royalty, betrayal, love, and revenge. \n\nI should mention the director, the production company, which is Baahubali Entertainments, and the music composer, which is S. Thaman. The film was a massive commercial success, being the first Indian film to gross over ₹1

In [5]:
response=model_with_structured.invoke("Provide details about the movie Bahubali")
response

Movie(title='Bahubali', year=2015, director='S.S. Rajamouli', rating=4.5)

### Message output alongside parsed structure

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year in which movie has released")
    director:str=Field(description="Director name of that movie")
    rating:float=Field(description="The movie rating out of 5")

model_with_structured=model.with_structured_output(Movie, include_raw=True)


response = model_with_structured.invoke("Provide details about the movie Bahubali")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Bahubali. Let me see what I need to do here. The tools provided include a Movie function that requires title, year, director, and rating. So, I need to gather that information for Bahubali.\n\nFirst, the title is obviously "Bahubali". I remember that the movie was released in 2015. The director is S.S. Rajamouli, right? He\'s a well-known director in the South Indian film industry. As for the rating, I think it\'s pretty popular, maybe around 4.5 out of 5 on IMDb. Let me double-check those details to make sure. Year 2015, director S.S. Rajamouli, and the rating is 7.7 on IMDb, which is a 4.5 star equivalent. Yeah, that sounds correct. I\'ll structure the function call with these parameters.\n', 'tool_calls': [{'id': 'kxwngeaaa', 'function': {'arguments': '{"director":"S.S. Rajamouli","rating":4.5,"title":"Bahubali","year":2015}', 'name': 'Movie'}, 'type': 'functio

### Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budgest in crore INR")

model_with_structured=model.with_structured_output(MovieDetails)

response=model_with_structured.invoke("Provide detail the movie Avengers endgame")
response

MovieDetails(title='Avengers: Endgame', year=2019, cast=[Actor(name='Robert Downey Jr.', role='Iron Man'), Actor(name='Chris Evans', role='Captain America'), Actor(name='Mark Ruffalo', role='Hulk'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Scarlett Johansson', role='Black Widow'), Actor(name='Jeremy Renner', role='Hawkeye')], genres=['Action', 'Sci-Fi', 'Adventure'], budget=356.0)

### TypeDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [ ]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    """A movie with details"""
    title: Annotated[str, ...,"The title of the movie"] 
    year: Annotated[int, ..., "he year in which movie has released"]
    director: Annotated[str, ..., "Director name of that movie"]
    rating: Annotated[float, ..., "The movie rating out of 5"]

model_with_structured=model.with_structured_output(Movie)
response = model_with_structured.invoke("Provide detail the movie Avengers Infinity war")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Infinity War',
 'year': 2018}

In [10]:

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budgest in crore INR")

model_with_structured=model.with_structured_output(MovieDetails)

response=model_with_structured.invoke("Provide detail the movie Avengers Infinity war")
response

{'budget': 200000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Paul Rudd', 'role': 'Ant-Man'},
  {'name': 'Benedict Cumberbatch', 'role': 'Doctor Strange'},
  {'name': 'Tom Holland', 'role': 'Spider-Man'},
  {'name': 'Thanos', 'role': 'Thanos'}],
 'genres': ['Action', 'Sci-Fi', 'Fantasy'],
 'title': 'Avengers: Infinity War',
 'year': 2018}

In [11]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Dataclasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [ ]:
# By pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str = Field(description="The name of the person")
    email:str=Field(description="The Email address of the person")
    phone:str = Field(description="The phone number of the person")

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from: Harsh Jyoriya, harshjyoriya@gmail.com, 6969696969"
    }]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: Harsh Jyoriya, harshjyoriya@gmail.com, 6969696969', additional_kwargs={}, response_metadata={}, id='0628c37d-76f2-40cf-a0a8-292d2d218d34'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "Harsh Jyoriya, harshjyoriya@gmail.com, 6969696969". \n\nFirst, I need to identify the different parts. The name is probably "Harsh Jyoriya" since that\'s the first part and looks like a full name. Next, the email address is "harshjyoriya@gmail.com". The last part is a phone number, which is "6969696969". \n\nI should check if all the required fields are present. The function ContactInfo requires name, email, and phone. All three are here. Now, I need to structure them into the function\'s parameters. The name is Harsh Jyoriya, email is the given address, and the phone number is the 10-digit number. \n\nI should make sure there\'s no extr

In [13]:
print(result["structured_response"])

name='Harsh Jyoriya' email='harshjyoriya@gmail.com' phone='6969696969'


In [15]:
# TypeDict
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name:str 
    email:str
    phone:str

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from: Harsh Jyoriya, harshjyoriya@gmail.com, 6969696969"
    }]
})

print(result['structured_response'])

{'name': 'Harsh Jyoriya', 'email': 'harshjyoriya@gmail.com', 'phone': '6969696969'}


In [16]:
# DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of the person"""
    name:str 
    email:str
    phone:str

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from: Harsh Jyoriya, harshjyoriya@gmail.com, 6969696969"
    }]
})

print(result['structured_response'])



ContactInfo(name='Harsh Jyoriya', email='harshjyoriya@gmail.com', phone='6969696969')
